# SimCLR Self-Supervised Pretraining on STL-10

Pretrains a ResNet-18 encoder on STL-10 unlabelled images using the SimCLR
contrastive objective (NT-Xent loss). The encoder is saved for FixMatch fine-tuning.

**Full run** (~90 min, Kaggle T4): `SMOKE_TEST = False`  
**Quick sanity check** (~2 min, CPU/low-disk): `SMOKE_TEST = True`

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://{token}@github.com/dima806/semi_supervised_image_clf.git

In [ ]:
!pip install git+https://{token}@github.com/dima806/semi_supervised_image_clf.git

In [ ]:
# Disable MLflow tracking globally (affects all internal training calls)
import mlflow

class _NoOpRun:
    def __enter__(self): return self
    def __exit__(self, *a): pass

mlflow.set_experiment = lambda *a, **kw: None
mlflow.start_run     = lambda *a, **kw: _NoOpRun()
mlflow.log_params    = lambda *a, **kw: None
mlflow.log_param     = lambda *a, **kw: None
mlflow.log_metric    = lambda *a, **kw: None
mlflow.log_metrics   = lambda *a, **kw: None
mlflow.end_run       = lambda *a, **kw: None
print('MLflow tracking disabled.')

## 1. Configuration

In [ ]:
# ── Set to True for a fast local sanity-check (uses < 200 MB of data) ──────
SMOKE_TEST = False
# ────────────────────────────────────────────────────────────────────────────

DATA_DIR       = '/kaggle/working/data/'
CHECKPOINT_DIR = '/kaggle/working/checkpoints/'

# Smoke-test caps (ignored when SMOKE_TEST = False)
SMOKE_MAX_UNLABELLED = 1_000   # images used for pretraining
SMOKE_MAX_LABELLED   = 200     # images used for linear probe
SMOKE_EPOCHS         = 2

print(f'SMOKE_TEST = {SMOKE_TEST}')

## 2. Environment setup

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU.')
print('Environment ready.')

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

from semi_supervised_image_clf.config import load_simclr_config
from semi_supervised_image_clf.model import ResNet18WithProjection
from semi_supervised_image_clf.simclr import SimCLRDataset, train_simclr

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}' + (f'  GPU: {torch.cuda.get_device_name(0)}' if DEVICE == 'cuda' else ''))

## 3. Download data

In [ ]:
# Only the unlabelled split is needed for pretraining (~2.5 GB full / ~25 MB smoke)
print('Downloading STL-10 unlabelled split...')
unlabelled_raw = datasets.STL10(root=DATA_DIR, split='unlabeled', download=True)
print(f'Total unlabelled images: {len(unlabelled_raw):,}')

if SMOKE_TEST:
    unlabelled_raw = Subset(unlabelled_raw, list(range(SMOKE_MAX_UNLABELLED)))
    print(f'Smoke test: using first {len(unlabelled_raw):,} images')

## 4. Build SimCLR loader & model

In [ ]:
config = load_simclr_config('semi_supervised_image_clf/config/simclr.yaml')

if SMOKE_TEST:
    config.training.epochs     = SMOKE_EPOCHS
    config.training.batch_size = 32
    config.training.warmup_epochs = 0
    config.data.num_workers    = 2
else:
    config.training.batch_size = 256
    config.data.num_workers    = 4

simclr_ds = SimCLRDataset(unlabelled_raw, input_size=config.model.input_size)
simclr_loader = DataLoader(
    simclr_ds,
    batch_size=config.training.batch_size,
    shuffle=True,
    num_workers=config.data.num_workers,
    pin_memory=(DEVICE == 'cuda'),
    drop_last=True,
)

model = ResNet18WithProjection(projection_dim=config.model.projection_dim)

print(f'Dataset:       {len(simclr_ds):,} images')
print(f'Batches/epoch: {len(simclr_loader):,}')
print(f'Epochs:        {config.training.epochs}')
print(f'Parameters:    {sum(p.numel() for p in model.parameters()):,}')

## 5. Pretraining

In [ ]:
model = train_simclr(
    model=model,
    loader=simclr_loader,
    config=config,
    checkpoint_dir=CHECKPOINT_DIR,
    smoke_test=False,   # epoch count already set above
)

print('\nSaved checkpoints:')
for f in sorted(Path(CHECKPOINT_DIR).glob('*.pt')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 6. Linear probe evaluation

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader as DL

# Download train + test only if not already on disk
transform = transforms.Compose([
    transforms.Resize((config.model.input_size, config.model.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_ds = datasets.STL10(root=DATA_DIR, split='train', transform=transform, download=True)
test_ds  = datasets.STL10(root=DATA_DIR, split='test',  transform=transform, download=True)

if SMOKE_TEST:
    train_ds = Subset(train_ds, list(range(SMOKE_MAX_LABELLED)))
    test_ds  = Subset(test_ds,  list(range(SMOKE_MAX_LABELLED)))

train_dl = DL(train_ds, batch_size=256, shuffle=True,  num_workers=2, pin_memory=(DEVICE=='cuda'))
test_dl  = DL(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=(DEVICE=='cuda'))

encoder = model.get_encoder().to(DEVICE)
for p in encoder.parameters():
    p.requires_grad_(False)

probe = nn.Linear(512, 10).to(DEVICE)
opt   = optim.Adam(probe.parameters(), lr=1e-3)
probe_epochs = 2 if SMOKE_TEST else 10

for epoch in range(1, probe_epochs + 1):
    probe.train()
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            h = encoder(imgs).flatten(1)
        loss = nn.functional.cross_entropy(probe(h), labels)
        opt.zero_grad(); loss.backward(); opt.step()

probe.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        h = encoder(imgs).flatten(1)
        correct += (probe(h).argmax(1) == labels).sum().item()
        total   += labels.size(0)

print(f'Linear probe accuracy: {correct / total:.4f}')

## 7. t-SNE of embeddings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

STL10_CLASSES = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
MAX_TSNE = 500 if SMOKE_TEST else 2000

feats, lbls = [], []
with torch.no_grad():
    for imgs, labels in test_dl:
        feats.append(encoder(imgs.to(DEVICE)).flatten(1).cpu().numpy())
        lbls.extend(labels.tolist())
        if len(lbls) >= MAX_TSNE:
            break

feats = np.concatenate(feats)[:MAX_TSNE]
lbls  = np.array(lbls[:MAX_TSNE])

perplexity = min(30, len(lbls) // 4)
emb = TSNE(n_components=2, random_state=42, perplexity=perplexity).fit_transform(feats)

fig, ax = plt.subplots(figsize=(10, 8))
for i, cls in enumerate(STL10_CLASSES):
    mask = lbls == i
    ax.scatter(emb[mask, 0], emb[mask, 1], label=cls, alpha=0.5, s=10)
ax.legend(markerscale=3, fontsize=9)
ax.set_title('t-SNE of SimCLR embeddings', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/tsne_simclr.png', dpi=150)
plt.show()

## 8. Output

| File | Description |
|---|---|
| `checkpoints/simclr_encoder.pt` | Encoder weights — input to `fixmatch_sweep.ipynb` |
| `checkpoints/simclr_full.pt` | Full model including projection head |

In [ ]:
for f in sorted(Path('/kaggle/working/').rglob('*.pt')):
    print(f'  {str(f.relative_to("/kaggle/working")):<45} {f.stat().st_size / 1e6:.1f} MB')